# Comparison file
**For comparing the custom implementations to the sklearn one**

Make sure to check what is expected for the custom implementation!

**TODO**
+ Add CV for the tree comp **DONE**
+ Add comp for the randomforest overall **DONE (sort of)**
+ try it out with some bigger datasets

### Helperfunctions

In [2]:
def load_dataset(csv_path: Optional[str] = None, 
                 target_col: Optional[str] = None, 
                 random_state: int = 42) -> Tuple[pd.DataFrame, str]:
    """
    Load dataset from CSV or fall back to California housing or synthetic regression.
    Returns (df, target_name).
    """
    if csv_path:
        df = pd.read_csv(csv_path)
        if target_col is None:
            target_col = df.columns[-1]
        return df, target_col

    # Try California housing (realistic regression) else synthetic
    try:
        data = fetch_california_housing(as_frame=True)
        df = data.frame.copy()
        # sklearn's fetch returns target in data.target; ensure a column name
        if "target" not in df.columns:
            df["target"] = data.target
            target_col = "target"
        else:
            target_col = df.columns[-1]
        return df, target_col
    except Exception:
        X, y = make_regression(n_samples=5000, n_features=10, noise=0.1, random_state=random_state)
        df = pd.DataFrame(X, columns=[f"X{i}" for i in range(X.shape[1])])
        df["target"] = y
        return df, "target"

def split_df(df: pd.DataFrame, 
             target_name: str, 
             test_size: float = 0.2, 
             random_state: int = 42) -> Tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
    X = df.drop(columns=[target_name])
    y = df[target_name]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)

    return X_train, X_test, y_train, y_test

def metrics_dict(y_true, y_pred) -> Dict[str, float]:
    return {
        "MSE": float(mean_squared_error(y_true, y_pred)),
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "MAE_pct": float(mean_absolute_percentage_error(y_true, y_pred)),
        "R2": float(r2_score(y_true, y_pred))
    }

# scikit-learn pipeline
def sklearn_pipeline(X_train: pd.DataFrame, 
                     y_train: pd.Series, 
                     X_test: pd.DataFrame, 
                     y_test: pd.Series,
                     alpha_for_pruning: Optional[float] = None,
                     random_state: int = 42) -> Dict[str, Any]:
    # initialize output dict
    out = {}

    # Create the classifier
    classifier = DecisionTreeRegressor(random_state=random_state)
    t0 = time.perf_counter()
    classifier.fit(X_train, y_train)
    fit_time = time.perf_counter() - t0

    t0 = time.perf_counter()
    y_pred = classifier.predict(X_test)
    pred_time = time.perf_counter() - t0

    out["sklearn_unpruned"] = {
        "model": classifier,
        "fit_time": fit_time,
        "predict_time": pred_time,
        "metrics": metrics_dict(y_test, y_pred)
    }

    if alpha_for_pruning:
        clf_pruned = DecisionTreeRegressor(random_state=random_state, ccp_alpha=alpha_for_pruning)
        t0 = time.perf_counter()
        clf_pruned.fit(X_train, y_train)
        pruned_fit_time = time.perf_counter() - t0

        t0 = time.perf_counter()
        y_pred_pruned = clf_pruned.predict(X_test)
        pruned_pred_time = time.perf_counter() - t0

        out["sklearn_pruned"] = {
            "alpha": alpha_for_pruning,
            "model": clf_pruned,
            "fit_time": pruned_fit_time,
            "predict_time": pruned_pred_time,
            "metrics": metrics_dict(y_test, y_pred_pruned)
        }
    else:
        out["sklearn_pruned"] = None

    return out

# custom pipeline
def custom_pipeline(CustomTreeClass, 
                    df_train: pd.DataFrame, 
                    target_name: str,
                    X_test: pd.DataFrame, 
                    y_test: pd.Series,
                    alpha_for_pruning: Optional[float] = None, 
                    random_state: int = 42,
                    categorical_features: List[str] = []) -> Dict[str, Any]:
    
    # initialize output dict
    out = {}

    # fit without pruning
    model = CustomTreeClass(random_state=random_state)

    t0 = time.perf_counter()
    model.fit(df_train, target_name, alpha=0.0, categorical_features=categorical_features)
    fit_time = time.perf_counter() - t0

    t0 = time.perf_counter()
    y_pred = model.predict(X_test)
    pred_time = time.perf_counter() - t0

    out["custom_unpruned"] = {
        "model": model,
        "fit_time": fit_time,
        "predict_time": pred_time,
        "metrics": metrics_dict(y_test, y_pred)
    }

    # fit with pruning if requested
    if alpha_for_pruning is not None and alpha_for_pruning > 0.0:
        t0 = time.perf_counter()
        model_pruned = CustomTreeClass(random_state=random_state)
        model_pruned.fit(df_train, target_name, alpha=alpha_for_pruning, categorical_features=categorical_features)
        prune_fit_time = time.perf_counter() - t0

        t0 = time.perf_counter()
        y_pred_pruned = model_pruned.predict(X_test)
        prune_pred_time = time.perf_counter() - t0

        out["custom_pruned"] = {
            "alpha": alpha_for_pruning,
            "model": model_pruned,
            "fit_time": prune_fit_time,
            "predict_time": prune_pred_time,
            "metrics": metrics_dict(y_test, y_pred_pruned)
        }
    else:
        out["custom_pruned"] = None

    return out

from sklearn.ensemble import RandomForestRegressor

# RandomForest pipelines
def sklearn_rf_pipeline(X_train: pd.DataFrame, 
                        y_train: pd.Series, 
                        X_test: pd.DataFrame, 
                        y_test: pd.Series,
                        n_estimators: int = 100,
                        random_state: int = 42) -> Dict[str, Any]:
    out = {}

    rf = RandomForestRegressor(
        n_estimators=n_estimators,
        random_state=random_state,
        n_jobs=-1
    )

    t0 = time.perf_counter()
    rf.fit(X_train, y_train)
    fit_time = time.perf_counter() - t0

    t0 = time.perf_counter()
    y_pred = rf.predict(X_test)
    pred_time = time.perf_counter() - t0

    out["sklearn_rf"] = {
        "model": rf,
        "fit_time": fit_time,
        "predict_time": pred_time,
        "metrics": metrics_dict(y_test, y_pred)
    }

    return out


# custom RandomForest pipeline
def custom_rf_pipeline(CustomForestClass, 
                       df_train: pd.DataFrame, 
                       target_name: str,
                       X_test: pd.DataFrame, 
                       y_test: pd.Series,
                       n_estimators: int = 100,
                       random_state: int = 42,
                       categorical_features: List[str] = []) -> Dict[str, Any]:
    out = {}

    rf_model = CustomForestClass(
        random_state=random_state
    )

    t0 = time.perf_counter()
    rf_model.fit(data=df_train, 
                 target_name=target_name, 
                 categorical_features=categorical_features,
                 nr_of_trees=n_estimators)

    fit_time = time.perf_counter() - t0

    t0 = time.perf_counter()
    y_pred = rf_model.predict(X_test)
    pred_time = time.perf_counter() - t0

    out["custom_rf"] = {
        "model": rf_model,
        "fit_time": fit_time,
        "predict_time": pred_time,
        "metrics": metrics_dict(y_test, y_pred)
    }

    return out


def compare_and_report(sk_results: Dict[str, Any], 
                       custom_results: Optional[Dict[str, Any]] = None):
    
    if sk_results.get("sklearn_unpruned"):
        print("\n=== scikit-learn: unpruned ===")
        su = sk_results["sklearn_unpruned"]
        print(f"fit_time: {su['fit_time']:.4f}s, predict_time: {su['predict_time']:.4f}s")
        print("metrics:", su["metrics"])

    if sk_results.get("sklearn_pruned"):
        sp = sk_results["sklearn_pruned"]
        print("\n=== scikit-learn: pruned (example) ===")
        print(f"alpha: {sp['alpha']}, fit_time: {sp['fit_time']:.4f}s, predict_time: {sp['predict_time']:.4f}s")
        print("metrics:", sp["metrics"])
    
    if sk_results.get("sklearn_rf"):
        srf = sk_results["sklearn_rf"]
        print("\n=== scikit-learn: Random Forest ===")
        print(f"fit_time: {srf['fit_time']:.4f}s, predict_time: {srf['predict_time']:.4f}s")
        print("metrics:", srf["metrics"])
    
    if custom_results.get("custom_unpruned"):
        cu = custom_results["custom_unpruned"]
        print("\n=== Custom tree: unpruned ===")
        print(f"fit_time: {cu['fit_time']:.4f}s, predict_time: {cu['predict_time']:.4f}s")
        print("metrics:", cu["metrics"])

    if custom_results.get("custom_pruned"):
        cp = custom_results["custom_pruned"]
        print("\n=== Custom tree: pruned ===")
        print(f"alpha: {cp['alpha']}, fit_time: {cp['fit_time']:.4f}s, predict_time: {cp['predict_time']:.4f}s")
        print("metrics:", cp["metrics"])

    if custom_results.get("custom_rf"):
        crf = custom_results["custom_rf"]
        print("\n=== Custom Random Forest ===")
        print(f"fit_time: {crf['fit_time']:.4f}s, predict_time: {crf['predict_time']:.4f}s")
        print("metrics:", crf["metrics"])

    # compact comparison table
    rows = []
    if sk_results.get("sklearn_unpruned"):
        rows.append({
            "model": "sklearn_unpruned",
            **su["metrics"],
            "fit_time": su["fit_time"],
            "predict_time": su["predict_time"]
        })
    if sk_results.get("sklearn_pruned"):
        rows.append({
            "model": f"sklearn_pruned_alpha={sk_results['sklearn_pruned']['alpha']}",
            **sk_results['sklearn_pruned']["metrics"],
            "fit_time": sk_results['sklearn_pruned']["fit_time"],
            "predict_time": sk_results['sklearn_pruned']["predict_time"]
        })
    
    if sk_results.get("sklearn_rf"):
        rows.append({
            "model": "sklearn_rf",
            **srf["metrics"],
            "fit_time": srf["fit_time"],
            "predict_time": srf["predict_time"]
        })

    if custom_results.get("custom_unpruned"):
        rows.append({
            "model": "custom_unpruned",
            **cu["metrics"],
            "fit_time": cu["fit_time"],
            "predict_time": cu["predict_time"]
        })

    if custom_results.get("custom_pruned"):
        rows.append({
            "model": f"custom_pruned_alpha={custom_results['custom_pruned']['alpha']}",
            **custom_results['custom_pruned']["metrics"],
            "fit_time": custom_results['custom_pruned']["fit_time"],
            "predict_time": custom_results['custom_pruned']["predict_time"]
        })
    
    if custom_results.get("custom_rf"):
        rows.append({
            "model": "custom_rf",
            **crf["metrics"],
            "fit_time": crf["fit_time"],
            "predict_time": crf["predict_time"]
        })

    df_comp = pd.DataFrame(rows).set_index("model")
    print("\n=== Summary comparison ===")
    print(df_comp.round(6).to_string())

### Output of Comparison

In [4]:
from typing import Any, Dict


def main(csv_path: Optional[str] = None, 
         target_col: Optional[str] = None,
         random_state: int = 42, 
         test_size: float = 0.2, 
         alpha_choice: Optional[float] = None,
         evaluate_alphas: Optional[List[float]] = None, 
         save_csv: Optional[str] = None,
         n_estimators: Optional[int] = 10) -> dict[str, Dict[str, Any]]:
    
    df, target_name = load_dataset(csv_path, target_col, random_state=random_state)
    print(f"Dataset: {df.shape[0]} rows, {df.shape[1]} cols. Target: {target_name}")

    X_train, X_test, y_train, y_test = split_df(df, target_name, test_size=test_size, random_state=random_state)
    df_train = pd.concat([X_train, y_train], axis=1) # necessary because implementation expects dataframe

    print("\nRunning scikit-learn pipeline...")
    sk_results = sklearn_pipeline(X_train, y_train, X_test, y_test, random_state=random_state, alpha_for_pruning=alpha_choice)


    print("\nRunning custom tree pipeline...")
    custom_results = custom_pipeline(RegressionTreeNico, df_train, target_name, X_test, y_test,
                                     alpha_for_pruning=alpha_choice, random_state=random_state)


    compare_and_report(sk_results, custom_results)

    # optional: evaluate multiple alphas for the custom tree and save CSV
    if evaluate_alphas:
        records = []
        for a in evaluate_alphas:
            print(f"\nEvaluating custom tree with alpha={a}")
            res = custom_pipeline(RegressionTreeNico, df_train, target_name, X_test, y_test,
                                  alpha_for_pruning=a, random_state=random_state)
            pr = res.get("custom_pruned") or res.get("custom_unpruned")
            records.append({
                "type": "custom",
                "alpha": a,
                **pr["metrics"],
                "fit_time": pr["fit_time"],
                "predict_time": pr["predict_time"]
            })

        for a in evaluate_alphas:
            print(f"\nEvaluating scikit-learn tree with alpha={a}")
            res = sklearn_pipeline(X_train, y_train, X_test, y_test, alpha_for_pruning=a, random_state=random_state)
            pr = res.get("sklearn_pruned") or res.get("sklearn_unpruned")
            records.append({
                "type": "scikit-learn",
                "alpha": a,
                **pr["metrics"],
                "fit_time": pr["fit_time"],
                "predict_time": pr["predict_time"]
            })    

        df_alphas = pd.DataFrame(records).set_index(["alpha", "type"]).sort_index()
        print("\nMetrics vs alpha:")
        print(df_alphas.round(6).to_string())
        if save_csv:
            df_alphas.to_csv(save_csv,)
            print(f"Saved alpha results to {save_csv}")

    print("\nRunning scikit-learn RandomForest pipeline...")
    sk_rf_results = sklearn_rf_pipeline(X_train, y_train, X_test, y_test, random_state=random_state,
                                        n_estimators=n_estimators)

    print("\nRunning custom RandomForest pipeline...")
    custom_rf_results = custom_rf_pipeline(RandomForestNico, df_train, target_name, X_test, y_test,
                                           random_state=random_state,
                                           n_estimators=n_estimators)

    compare_and_report(sk_rf_results, custom_rf_results)


    return {"sklearn tree": sk_results, 
            "custom tree": custom_results,
            "sklearn forest": sk_rf_results, 
            "custom forest": custom_rf_results
            }




if __name__ == "__main__":

    main(target_col="target", 
         #evaluate_alphas=[0.0, 0.1],#, 1, 10, 100], 
         #save_csv="alpha_results.csv",
         n_estimators=1000)

Dataset: 20640 rows, 10 cols. Target: target

Running scikit-learn pipeline...

Running custom tree pipeline...


TypeError: RegressionTreeNico.fit() got an unexpected keyword argument 'alpha'

## Debugarea

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# for testing and comparison
import time
from sklearn.tree import DecisionTreeRegressor
from sklearn.datasets import make_regression, fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error
from sklearn.preprocessing import FunctionTransformer, StandardScaler, OneHotEncoder, LabelEncoder

# for, well typing
from typing import Literal, Optional, Dict, Any, Self, Tuple, List

from customRegressionTreeForest import *

from sklearn.model_selection import ShuffleSplit, GridSearchCV, ShuffleSplit, StratifiedKFold
from sklearn.pipeline import Pipeline
from customRegressionTreeForest import *

def metrics_dict(y_true, y_pred) -> Dict[str, float]:
    return {
        "MSE": float(mean_squared_error(y_true, y_pred)),
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "MAE_pct": float(mean_absolute_percentage_error(y_true, y_pred)),
        "R2": float(r2_score(y_true, y_pred))
    }

def split_df(df: pd.DataFrame, 
             target_name: str, 
             test_size: float = 0.2, 
             random_state: int = 42) -> Tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
    X = df.drop(columns=[target_name])
    y = df[target_name]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)

    return X_train, X_test, y_train, y_test

def evaluate_run(name, task_type, y_true, y_pred):
    return dict(
            id=id,
            name=name,
            task_type = task_type,
            r2=r2_score(y_true, y_pred), 
            mse = mean_squared_error(y_true, y_pred), 
            mae = mean_absolute_error(y_true, y_pred), 
            mape = mean_absolute_percentage_error(y_true, y_pred)
        )

data = fetch_california_housing(as_frame=True)
df = data.frame.copy()
# sklearn's fetch returns target in data.target; ensure a column name
if "target" not in df.columns:
    df["target"] = data.target
    target_col = "target"
else:
    target_col = df.columns[-1]

df

X_train, X_test, y_train, y_test = split_df(df, target_name = "target")

param_grid = {"classifier__max_depth": [None,3,5,7]}

pipe = Pipeline([
    #("preprocess", StandardScaler()),
    ("classifier", RegressionTreeNico())
])

# gridwork whoop
grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    scoring="r2",
    refit=True,
    return_train_score=False
)

grid.fit(X_train, y_train)
best_pipe = grid.best_estimator_
best_params = grid.best_params_

y_pred = best_pipe.predict(X_test)

metrics_dict(y_test, y_pred)




{'MSE': 3.7546743484021496e-06,
 'MAE': 0.0001965026301218989,
 'MAE_pct': 0.00035319097111235283,
 'R2': 0.999997134730904}

In [ ]:
best_pipe
# attention, big surprise coming up...

# with scaling
#{'MSE': 3.7992479918130005e-06,
# 'MAE': 0.00019940960686608386,
# 'MAE_pct': 0.00035383250428220816,
# 'R2': 0.9999971007158415}

# without scaling
#{'MSE': 3.7546743484021496e-06,
# 'MAE': 0.0001965026301218989,
# 'MAE_pct': 0.00035319097111235283,
# 'R2': 0.999997134730904}

# almost no difference as expected for trees

,steps,"[('preprocess', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,min_instances,2
,max_depth,None
,max_features,None
,features_to_choose,None


In [5]:
param_grid = {"classifier__max_depth": [None,3,5,7]}

pipe = Pipeline([
   # ("preprocess", StandardScaler()),
    ("classifier", DecisionTreeRegressor())
])

# gridwork whoop
grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    scoring="r2",
    refit=True,
    return_train_score=False
)

grid.fit(X_train, y_train)
best_pipe = grid.best_estimator_
best_params = grid.best_params_

y_pred = best_pipe.predict(X_test)

metrics_dict(y_test, y_pred)

{'MSE': 5.0055717538759645e-06,
 'MAE': 0.0002902180232595946,
 'MAE_pct': 0.0004602048905700486,
 'R2': 0.9999961801454073}

In [12]:
data = fetch_california_housing(as_frame=True)
df = data.frame.copy()
# sklearn's fetch returns target in data.target; ensure a column name
if "target" not in df.columns:
    df["target"] = data.target
    target_col = "target"
else:
    target_col = df.columns[-1]

X = df.drop(columns=[target_col])
y = df[target_col]


preprocessing_variants = {
    "None": None,
    "scaler": StandardScaler()
}

classifiers = {
   "DecisionTreeRegressor": DecisionTreeRegressor(max_depth=None, random_state=21),
   "RegressionTreeNico": RegressionTreeNico(max_depth=None, random_state=21)
}

prep_name = "StandardScaler()"
results = []
cv = ShuffleSplit(n_splits=10, test_size=0.3, random_state=21)

for clf_name, clf in classifiers.items():
    for prep_name, prepr in preprocessing_variants.items():
        # to capture all the results
        fold_metrics = []
        
        # iterate through al the folds
        for fold, (train_idx, test_idx) in enumerate(cv.split(X, y)):
            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]

            if prep_name == "None":
                pipe = Pipeline([("classifier", clf)])
            
            else:
                pipe = Pipeline([
                    ("preprocess", prepr),
                    ("classifier", clf)
                ])

            pipe.fit(X_train, y_train)
            y_pred = pipe.predict(X_test) 

            eval_dict = evaluate_run(
                name=f"{prep_name} + {clf_name} (fold {fold+1})",
                y_true=y_test,
                y_pred=y_pred,
                task_type="regression"
            )
            fold_metrics.append(eval_dict)
            
        # Aggregate across folds for stats 
        df_metrics = pd.DataFrame(fold_metrics)
        agg = df_metrics[['r2', 'mse', 'mae', 'mape']].agg(['mean', 'std'])
        results.append({
            "Preprocessing": prep_name,
            "Classifier": clf_name,
            "r2_mean": agg.loc['mean', 'r2'],
            "r2_std": agg.loc['std', 'r2'],
            "mse_mean": agg.loc['mean', 'mse'],
            "mse_std": agg.loc['std', 'mse'],
            "mae_mean": agg.loc['mean', 'mae'],
            "mae_std": agg.loc['std', 'mae'],
            "mape_mean": agg.loc['mean', 'mape'],
            "mape_std": agg.loc['std', 'mape']
        })

results_df = pd.DataFrame(results)

In [13]:
results_df

,Preprocessing,Classifier,r2_mean,r2_std,mse_mean,mse_std,mae_mean,mae_std,mape_mean,mape_std
0,None,DecisionTreeRegressor,0.999999,4.836381e-07,1.600396e-06,6.600464e-07,0.000288,0.000020,0.000217,0.000051
1,scaler,DecisionTreeRegressor,0.999999,4.852690e-07,1.602576e-06,6.624772e-07,0.000288,0.000021,0.000217,0.000052
2,None,RegressionTreeNico,0.999999,5.123089e-07,9.931541e-07,6.965815e-07,0.000198,0.000018,0.000147,0.000050
3,scaler,RegressionTreeNico,0.999999,5.124367e-07,9.898782e-07,6.967889e-07,0.000198,0.000018,0.000147,0.000049
